# ISIC skin lesions — test-set CAM audit

This notebook is the visual companion to the [full benchmark](../results/isic/RESULTS.md).
It shows **three frozen test examples per architecture** using the exact same
center-cropped input for CE, FCAL V1, FCAL V2, and—where available—Universal FCAL.

How to read each panel:

- green/red text means the predicted class is correct/incorrect;
- the white contour is the annotated foreground;
- `inside` is the share of positive **predicted-class CAM** mass inside that contour;
- CAM colors are normalized within each panel and use `jet` (blue = low,
  red = high).

The examples are selected reproducibly from saved test arrays: first a strong
localization comparison where CE and V2 are both correct, then a CE→V2
classification rescue when one exists, then another localization improvement.
This is a qualitative audit, not an accuracy estimate; use the full result tables
for quantitative claims.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ASSETS = ROOT / "assets" / "result_showcases" / "isic"
manifest = pd.read_csv(ASSETS / "examples.csv")
print(f"Loaded {len(manifest)} frozen examples from {manifest['architecture'].nunique()} architectures")
manifest[["family", "architecture", "case_rank", "image_id", "true_class"]]

Loaded 18 frozen examples from 6 architectures


,family,architecture,case_rank,image_id,true_class
0,CNN,ResNet-50,1,24948,AK
1,CNN,ResNet-50,2,24579,SCC
2,CNN,ResNet-50,3,24358,BKL
3,CNN,DenseNet-121,1,24324,BKL
4,CNN,DenseNet-121,2,25368,AK
5,CNN,DenseNet-121,3,24408,BKL
6,CNN,EfficientNet-B0,1,12,NV
7,CNN,EfficientNet-B0,2,13,MEL
8,CNN,EfficientNet-B0,3,25368,AK
9,ViT,ViT-Tiny/16,1,24383,BKL


## CNNs

These CAMs are exact final-feature class activation maps: the classifier's own
linear weights projected back onto the final convolutional feature grid.

### ResNet-50

![ResNet-50 test cases](../assets/result_showcases/isic/resnet50_test_cases.png)

### DenseNet-121

![DenseNet-121 test cases](../assets/result_showcases/isic/densenet121_test_cases.png)

### EfficientNet-B0

![EfficientNet-B0 test cases](../assets/result_showcases/isic/efficientnet-b0_test_cases.png)

## Vision Transformers

The ViT runs use average-pooled patch tokens, so their class-specific patch CAMs
decompose the actual classifier score rather than substituting a generic
self-attention visualization.

### ViT-Tiny/16

![ViT-Tiny/16 test cases](../assets/result_showcases/isic/vit-tiny-p16-avg_test_cases.png)

### ViT-Small/16

![ViT-Small/16 test cases](../assets/result_showcases/isic/vit-small-p16-avg_test_cases.png)

### ViT-Base/16

![ViT-Base/16 test cases](../assets/result_showcases/isic/vit-base-p16-avg_test_cases.png)

## Selected-example audit table

The table below exposes every displayed prediction and inside-foreground value.
Because these examples were deliberately selected for visual diagnosis, do not
average this table or present it as benchmark performance.

In [2]:
display_columns = [
    column for column in manifest.columns
    if column in {"architecture", "case_rank", "image_id", "true_class"}
    or column.endswith("_correct")
    or column.endswith("_inside_fraction")
]
manifest[display_columns].style.format({
    column: "{:.1%}" for column in display_columns if column.endswith("_inside_fraction")
})

,architecture,case_rank,image_id,true_class,ce_correct,ce_inside_fraction,fcal_v1_correct,fcal_v1_inside_fraction,fcal_v2_correct,fcal_v2_inside_fraction
0,ResNet-50,1,24948,AK,True,5.3%,True,22.9%,True,50.3%
1,ResNet-50,2,24579,SCC,False,93.8%,False,92.9%,True,95.9%
2,ResNet-50,3,24358,BKL,True,38.3%,True,47.1%,True,78.0%
3,DenseNet-121,1,24324,BKL,True,41.2%,True,48.2%,True,71.1%
4,DenseNet-121,2,25368,AK,False,41.4%,False,45.4%,True,63.5%
5,DenseNet-121,3,24408,BKL,True,27.5%,True,28.6%,True,53.5%
6,EfficientNet-B0,1,12,NV,True,32.7%,True,37.2%,True,41.0%
7,EfficientNet-B0,2,13,MEL,False,85.9%,False,86.5%,True,81.9%
8,EfficientNet-B0,3,25368,AK,True,58.9%,True,62.5%,True,67.9%
9,ViT-Tiny/16,1,24383,BKL,True,52.9%,True,58.1%,True,61.5%


## Artifact provenance

The committed PNGs were reconstructed from the real best-validation checkpoints.
The accompanying `examples.csv` manifest freezes every displayed test ID,
prediction, correctness flag, confidence, and inside-foreground CAM value, so
this notebook remains auditable without downloading checkpoints or datasets.